In [ ]:
!pip install -q \
pypdf==5.1.0 \
faiss-cpu==1.8.0.post1 \
sentence-transformers==3.0.1 \
transformers==4.44.2 \
google-generativeai==0.8.5 \
gradio==5.38.2

In [7]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Loaded successfully!")

Loaded successfully!


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [8]:
from google.colab import files
uploaded = files.upload()

In [9]:
import os
from pypdf import PdfReader
documents = []
for filename in os.listdir():
    if filename.endswith(".pdf"):
        reader = PdfReader(filename)
        for page_no, page in enumerate(reader.pages):
            text = page.extract_text()
            if text:
                documents.append({
                    "source": filename,
                    "page": page_no + 1,
                    "text": text
                })
print(f"Loaded {len(documents)} pages.")
print(f"Loaded {len(documents)} pages from {len(set(d['source'] for d in documents))} PDFs")

Loaded 88 pages.
Loaded 88 pages from 9 PDFs


In [10]:
documents[0]

{'source': 'UG_RULE_BOOK.pdf',
 'page': 1,
 'text': ' \n \n \n \nINDIAN INSTITUTE OF TECHNOLOGY BOMBAY \n \n \nRules & Regulations  \nfor Undergraduate Programmes \n \n \n \nApplicable to the B.Tech., B.S., B.Des.,   \nDual Degree students admitted from the  \nAcademic Year 2007 - 2008 \n \n \n \n \n \n \n \n \n \n \nUpdated: June, 2025   \n'}

In [11]:
def chunk_text(text, chunk_size=350, overlap=75):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

In [12]:
chunked_documents = []
chunk_id = 0
for doc in documents:
    chunks = chunk_text(doc["text"])
    for chunk in chunks:
        chunked_documents.append({
            "id": chunk_id,
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"]
        })
        chunk_id += 1
print(f"Total chunks: {len(chunked_documents)}")

Total chunks: 155


In [13]:
chunked_documents[0]

{'id': 0,
 'text': 'INDIAN INSTITUTE OF TECHNOLOGY BOMBAY Rules & Regulations for Undergraduate Programmes Applicable to the B.Tech., B.S., B.Des., Dual Degree students admitted from the Academic Year 2007 - 2008 Updated: June, 2025',
 'source': 'UG_RULE_BOOK.pdf',
 'page': 1}

In [14]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [chunk["text"] for chunk in chunked_documents]
print("Total texts:", len(texts))
embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Total texts: 155


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

In [15]:
print(embeddings.shape)

(155, 384)


In [16]:
import faiss
import numpy as np
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
print("Total vectors:", index.ntotal)
faiss.write_index(index, "iitb_vector.index")


Total vectors: 155


In [17]:
import pickle
with open("metadata.pkl", "wb") as f:
    pickle.dump(chunked_documents, f)

In [18]:
index = faiss.read_index("iitb_vector.index")
with open("metadata.pkl", "rb") as f:
    metadata = pickle.load(f)
print(index.ntotal)
print(len(metadata))

155
155


In [19]:
def retrieve(query, top_k=3):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    scores, indices = index.search(query_embedding, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "source": metadata[idx]["source"],
            "page": metadata[idx]["page"],
            "text": metadata[idx]["text"]
        })
    return results

In [20]:
results = retrieve("Can I pursue two minors?")
for i, result in enumerate(results, start=1):
    print(f"\nResult {i}")
    print(f"Score : {result['score']:.4f}")
    print(f"Source: {result['source']}")
    print(f"Page  : {result['page']}")
    print(result["text"][:500])


Result 1
Score : 0.4567
Source: Policy Newsletter.pdf
Page  : 2
The revised rule broadens the scope, allowing all students in the 8th semester or 10th semester to register for up to 54 credits, regardless of how many credits they have remaining. This would still require approval from the Faculty Adviser. Rationale: The previous rule only beneﬁts students with 54 or fewer credits left to complete, while those with more credits are restricted by their category limits . The new rule aims to level the playing ﬁeld by allowing all students in their ﬁnal semester 

Result 2
Score : 0.4313
Source: UG_RULE_BOOK.pdf
Page  : 13
13 Move to Index or all of the additional time for extra - curricular activities (including social work) if they so desire, and gain hands-on administrative/ managerial/ aesthetic skills or sensitivity towards social issues. Since seats available in such additional courses are often limited and the competition severe, stu- dents aspiring to do these additional courses ha

In [21]:
from google.colab import userdata
import google.generativeai as genai

API_KEY = userdata.get("GEMINI_API_KEY")

genai.configure(api_key=API_KEY)

llm = genai.GenerativeModel("gemini-3.5-flash")

print(" Gemini configured successfully!")




 Gemini configured successfully!


In [28]:
import google.generativeai as genai
genai.configure(api_key=API_KEY)
# Initializing the models exactly as you styled them
llm_model = genai.GenerativeModel("gemini-3.5-flash") # Note: Adjusted to valid model naming
model_gemini = genai.GenerativeModel("gemini-3.5-flash")
# To print text, we generate content first using your model variable
response = model_gemini.generate_content("Hello! Introduce yourself briefly.")
print(response.text)

Hello! I'm Gemini, an AI assistant created by Google. 

I'm here to help you with a wide variety of tasks—whether you want to brainstorm ideas, write or edit text, learn something new, write code, or just have a friendly chat. 

How can I help you today?


In [24]:
def create_prompt(question, retrieved_documents):
    context = "\n\n".join([doc["text"] for doc in retrieved_documents])
    prompt = f"""You are IITB Insti-Assist.
Answer ONLY using the retrieved context.
If the answer is fully supported by the context, answer confidently.
If the context does NOT contain enough information, reply ONLY:
"I don't know based on the provided documents."
Do not mix supported and unsupported information.
Always mention the source document(s) and page number(s).
Context:
{context}
Question: {question}
Answer:"""
    return prompt

In [29]:
def answer_question(question):
    retrieved = retrieve(question, top_k=3)
    if not retrieved:
        return "I don't know based on the provided documents."
    prompt = create_prompt(question, retrieved)
    try:
        response = llm_model.generate_content(prompt)
        if hasattr(response, "text"):
            return response.text
        else:
            return "No response generated."
    except Exception as e:
        return f"Error: {e}"

In [26]:
print(answer_question("Can I pursue two minors?"))
print(answer_question("Who is the Prime Minister of Canada?"))

Yes, you can pursue two minors. Under the revised policy, all students, regardless of category, are allowed to opt for and pursue more than one minor (including two minors, or an Honour and a minor) if their timetable permits. Students are encouraged to consult with their Faculty Advisers before doing so, as completing more than one minor involves a significant course overload.

**Sources:**
* **First page of the context** (Section 3: Dual Minor Policy Update: Expanded Access for Students)
* **Page 14** (Section 2.5.3: More than one minor for students)
I don't know based on the provided documents.


In [45]:
def chatbot(question):
    retrieved = retrieve(question, top_k=2)
    if not retrieved:
        return (
            "I don't know based on the provided documents.",
            "No supporting documents found."
        )
    prompt = create_prompt(question, retrieved)
    response = model_gemini.generate_content(prompt)
    answer = response.text
    sources = ""
    seen = set()
    for r in retrieved:
        source = f"• {r['source']} (Page {r['page']})"
        if source not in seen:
            sources += source + "\n"
            seen.add(source)
    return answer, sources

In [43]:
import gradio as gr
# Refined CSS for custom styling matching the blue-violet vibe
custom_css = """
.gradio-container {
    max-width: 1000px !important;
    margin: auto;
}
h1 {
    text-align: center;
    background: linear-gradient(45deg, #4f46e5, #7c3aed);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}
#answer-box textarea {
    font-size: 16px;
    font-weight: 500;
}
#source-box textarea {
    font-size: 14px;
    font-family: monospace;
}
"""
with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="indigo",     # Sleek Deep Blue/Indigo base
        secondary_hue="violet"    # Radiant Violet secondary accents
    ),
    css=custom_css
) as demo:

    gr.Markdown(
        """
        # IITB Insti-Assist

        ### Your AI assistant for IIT Bombay academic information

        Ask questions about:
        - Academic rules
        - UG/PG regulations
        - Courses and minors
        - Institute policies

        *Powered by official IIT Bombay documents using Retrieval-Augmented Generation (RAG).*
        """
    )

    with gr.Row():
        with gr.Column(scale=2):
            question = gr.Textbox(
                label="Your Question",
                placeholder="Example: Can students opt for more than one minor?",
                lines=3
            )
            submit = gr.Button(
                "✨ Ask Insti-Assist",
                variant="primary"
            )
            examples = gr.Examples(
                examples=[
                    ["Can students opt for more than one minor?"],
                    ["What is the role of a faculty adviser?"],
                    ["What are the requirements for a dual degree?"],
                    ["How many credits are required for graduation?"]
                ],
                inputs=question
            )

        with gr.Column(scale=3):
            answer = gr.Textbox(
                label="Answer",
                lines=8,
                elem_id="answer-box",
                interactive=False  # Keeps output clean from user modification
            )
            sources = gr.Textbox(
                label="Sources Used",
                lines=5,
                elem_id="source-box",
                interactive=False  # Keeps sources display strictly structural
            )

    # Clicking the button triggers the RAG backend
    submit.click(
        fn=chatbot,
        inputs=question,
        outputs=[answer, sources]
    )

    # Pressing Enter inside the text box also triggers the RAG backend
    question.submit(
        fn=chatbot,
        inputs=question,
        outputs=[answer, sources]
    )

    gr.Markdown(
        """
        ---
        **Disclaimer:** This assistant answers only from retrieved IIT Bombay official documents.
        If information is unavailable, it will respond:
        "I don't know based on the provided documents."
        """
    )
    demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://41966399331c5c994f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://eb0a6b3b69448288e3.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://41966399331c5c994f.gradio.live
